# 📖 Notebook 5: Production Best Practices

Welcome to the final notebook! So far we've covered Kafka fundamentals, partitioning, delivery guarantees, and stream processing. Now let's look at what separates a **toy Kafka setup** from a **production-ready one**.

This notebook follows a **bad → best practice progression**: for each topic we first show the naive approach (what you might write first), explain why it fails in production, then show the improved version.

## 🎯 What You'll Learn

1. **Schema validation** — bad: send any JSON → best: validate with a schema before publishing
2. **Producer batching & compression** — bad: default settings → best: tuned for throughput
3. **Dead Letter Queue (DLQ)** — bad: crash on poison message → best: send bad messages to a DLQ
4. **Log retention & compaction** — bad: keep everything forever → best: policy per use case
5. **Replication & `min.insync.replicas`** — bad: single replica → best: durable, tolerates failures
6. **Consumer lag** — bad: no monitoring → best: measure and alert on lag


## 🛠️ Setup

1. Kafka running via Docker Compose:
   ```bash
   cd 03-technologies/messaging/kafka
   docker-compose up -d
   ```

2. Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

> **Note on replication:** Our local lab runs a **single broker**, so replication-factor=1 is all we can demonstrate practically. For those sections we'll show the *configuration* you'd use in a multi-broker cluster and explain the behavior — the code runs on a single broker, but the settings are production-realistic.


In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic, ConfigResource, RESOURCE_TOPIC
import json, time

KAFKA_CONFIG = {'bootstrap.servers': 'localhost:9092'}
admin = AdminClient(KAFKA_CONFIG)

try:
    meta = admin.list_topics(timeout=5)
    print(f"✅ Connected to Kafka. {len(meta.topics)} topics exist.")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   Run: cd 03-technologies/messaging/kafka && docker-compose up -d")

def ensure_topic(name, partitions=1, config=None):
    t = NewTopic(name, num_partitions=partitions, replication_factor=1, config=config or {})
    fs = admin.create_topics([t])
    for _, f in fs.items():
        try: f.result()
        except Exception: pass


---

## 1️⃣ Schema Validation

### 😬 Bad: Publish any JSON

Without a schema, producers can send *anything*. The moment a developer typos a field name (`userId` vs `user_id`), or ships a breaking change, every consumer crashes.


In [ ]:
ensure_topic('orders-naive')

producer = Producer(KAFKA_CONFIG)

# Different "versions" of the same event — no one enforces shape
messy_events = [
    {'order_id': 'A1', 'user_id': 'u1', 'amount': 10.0},
    {'orderId': 'A2', 'userId': 'u2', 'amt': 20.0},        # 🚨 renamed fields
    {'order_id': 'A3', 'user_id': 'u3'},                    # 🚨 missing 'amount'
    {'order_id': 'A4', 'user_id': 'u4', 'amount': 'free'},  # 🚨 wrong type
]

for ev in messy_events:
    producer.produce('orders-naive', value=json.dumps(ev).encode())
producer.flush()
print("⚠️ All 4 events published — no validation. Consumers will crash on 3 of them.")


### ✅ Best: Validate against a schema before publishing

We'll use a tiny hand-rolled validator so we don't need extra dependencies. In production you'd use the **`jsonschema`** library or — even better — **Confluent Schema Registry** with Avro / JSON Schema / Protobuf. The idea is identical: **reject bad data at the edge**, not deep inside a consumer.

Key benefits:
- Bad data never enters the topic → all downstream consumers are safer.
- Schemas act as a **contract** between producer and consumer teams.
- With a registry, consumers can read the schema a message was written with, enabling safe schema evolution.


In [ ]:
ORDER_SCHEMA = {
    'order_id': str,
    'user_id': str,
    'amount': (int, float),
}

def validate(event, schema):
    for field, expected_type in schema.items():
        if field not in event:
            raise ValueError(f"missing required field: {field}")
        if not isinstance(event[field], expected_type):
            raise ValueError(f"field {field!r} expected {expected_type}, got {type(event[field]).__name__}")
    return True

ensure_topic('orders-validated')
producer = Producer(KAFKA_CONFIG)

accepted, rejected = 0, 0
for ev in messy_events:
    try:
        validate(ev, ORDER_SCHEMA)
        producer.produce('orders-validated', value=json.dumps(ev).encode())
        accepted += 1
        print(f"  ✅ accepted: {ev}")
    except ValueError as err:
        rejected += 1
        print(f"  ❌ rejected: {ev}  <- {err}")

producer.flush()
print(f"\n📊 {accepted} accepted, {rejected} rejected before they could pollute the topic")


---

## 2️⃣ Producer Batching & Compression

### 😬 Bad: flush after every message

`flush()` forces a network round-trip. Calling it after every `produce()` collapses throughput.

### ✅ Best: let the producer batch, and compress the batches

Two knobs do most of the work:

| Setting | What it does | Typical value |
|---|---|---|
| `linger.ms` | Wait this long to accumulate a batch before sending | 5–50 ms |
| `batch.size` | Target batch size in bytes | 16 KB – 1 MB |
| `compression.type` | Compress each batch | `lz4`, `zstd`, `snappy` |

**Why it works:** sending 1,000 messages in one 200 KB compressed batch is dramatically cheaper than sending 1,000 separate 200-byte requests. Kafka is built for this pattern.


In [ ]:
ensure_topic('perf-demo', partitions=4)

N = 2_000
payload = ('x' * 200).encode()  # 200-byte messages

# --- BAD: flush after every message
bad = Producer(KAFKA_CONFIG)
start = time.time()
for i in range(N):
    bad.produce('perf-demo', value=payload)
    bad.flush()  # 🚨 round-trip per message
bad_elapsed = time.time() - start
print(f"😬 Bad (flush per message): {N} msgs in {bad_elapsed:.2f}s -> {N/bad_elapsed:,.0f} msg/s")

# --- GOOD: batch + compress, flush once at the end
good = Producer({
    **KAFKA_CONFIG,
    'linger.ms': 20,
    'batch.size': 64 * 1024,
    'compression.type': 'lz4',
})
start = time.time()
for i in range(N):
    good.produce('perf-demo', value=payload)
good.flush()
good_elapsed = time.time() - start
print(f"✅ Good (batch + lz4):       {N} msgs in {good_elapsed:.2f}s -> {N/good_elapsed:,.0f} msg/s")

print(f"\n🚀 Speedup: {bad_elapsed/good_elapsed:.1f}x faster with batching + compression")


---

## 3️⃣ Dead Letter Queue (DLQ) — Handling Poison Messages

### 😬 Bad: crash on a bad message

A "poison message" is one the consumer can't process — malformed JSON, unknown type, failing business rule. If you let the exception propagate and Kafka redelivers the same message, you get an infinite crash loop that blocks *every* other message in that partition.

```python
while True:
    msg = consumer.poll(1.0)
    event = json.loads(msg.value())  # one bad byte and the consumer is stuck forever
    process(event)
```

### ✅ Best: route bad messages to a Dead Letter Queue

The **DLQ** is just another Kafka topic. When processing fails, the consumer:

1. Writes the failed message (+ error details as headers) to the DLQ topic.
2. Commits its offset on the main topic so it moves on.
3. A human or an automated tool inspects the DLQ later (replay, alert, discard).

This keeps the main pipeline flowing and makes failures **visible** instead of hidden crashes.


In [ ]:
ensure_topic('orders-input')
ensure_topic('orders-dlq')

# Seed some good + some poison messages
seed = Producer(KAFKA_CONFIG)
seed.produce('orders-input', value=json.dumps({'order_id': 'A1', 'amount': 10}).encode())
seed.produce('orders-input', value=b'not-json-at-all')  # poison
seed.produce('orders-input', value=json.dumps({'order_id': 'A2', 'amount': 'free'}).encode())  # bad type
seed.produce('orders-input', value=json.dumps({'order_id': 'A3', 'amount': 30}).encode())
seed.flush()
print("✅ Seeded 4 messages (2 good, 2 poison)\n")

consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'dlq-demo', 'auto.offset.reset': 'earliest',
                     'enable.auto.commit': False})
consumer.subscribe(['orders-input'])
dlq = Producer(KAFKA_CONFIG)

def process(event):
    return event['amount'] * 1.1  # requires numeric amount

processed, dead = 0, 0
empty = 0
while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1; continue
    if msg.error():
        continue
    empty = 0

    try:
        event = json.loads(msg.value().decode())
        result = process(event)
        print(f"  ✅ processed order {event['order_id']} -> total {result:.2f}")
        processed += 1
    except Exception as err:
        dlq.produce(
            'orders-dlq',
            value=msg.value(),
            headers=[
                ('error', str(err).encode()),
                ('source_topic', msg.topic().encode()),
                ('source_partition', str(msg.partition()).encode()),
                ('source_offset', str(msg.offset()).encode()),
            ],
        )
        print(f"  🪦 poison message sent to DLQ (offset {msg.offset()}): {err}")
        dead += 1

    consumer.commit(message=msg)  # always commit — whether DLQ'd or processed

dlq.flush()
consumer.close()
print(f"\n📊 Processed {processed}, DLQ'd {dead} — main pipeline never blocked")


---

## 4️⃣ Log Retention & Compaction

Kafka is a **log** — messages don't disappear when consumed. You have to tell Kafka how long to keep them.

### 😬 Bad: keep everything forever

Default retention is 7 days, but many teams bump it to "forever" just in case. Disk fills up, rebalances slow down, broker restarts take hours.

### ✅ Best: pick a retention policy that matches the use case

Kafka offers two **cleanup policies**:

| Policy | What it keeps | Use it for |
|---|---|---|
| `delete` (default) | Messages younger than `retention.ms` | Event streams (clicks, logs) — old events are worthless |
| `compact` | Only the **latest** message per key | Stateful topics (user profile, product catalog) — you only care about the current value |
| `compact,delete` | Latest per key **and** young enough | Best of both — e.g., session state with TTL |

**Log compaction** is Kafka's killer feature for state: you can replay a compacted topic and reconstruct the current state of the world, regardless of how old the data is.


In [ ]:
ensure_topic('clicks-stream', config={
    'cleanup.policy': 'delete',
    'retention.ms': str(60 * 60 * 1000),  # 1 hour
})

ensure_topic('user-profiles', config={
    'cleanup.policy': 'compact',
    'min.cleanable.dirty.ratio': '0.1',
    'segment.ms': '60000',
})

def show_config(topic):
    resources = [ConfigResource(RESOURCE_TOPIC, topic)]
    result = admin.describe_configs(resources)
    cfg = list(result.values())[0].result()
    keys = ['cleanup.policy', 'retention.ms', 'min.cleanable.dirty.ratio']
    print(f"\n📋 Config for '{topic}':")
    for k in keys:
        if k in cfg:
            print(f"   {k} = {cfg[k].value}")

show_config('clicks-stream')
show_config('user-profiles')

producer = Producer(KAFKA_CONFIG)
updates = [
    ('user-42', {'name': 'Alice', 'plan': 'free'}),
    ('user-42', {'name': 'Alice', 'plan': 'pro'}),
    ('user-42', {'name': 'Alice', 'plan': 'enterprise'}),
    ('user-99', {'name': 'Bob', 'plan': 'free'}),
]
for key, val in updates:
    producer.produce('user-profiles', key=key.encode(), value=json.dumps(val).encode())
producer.flush()

print("\n💡 After compaction runs, 'user-profiles' will keep ONLY the latest value per key:")
print("   user-42 -> plan=enterprise  (older entries are garbage-collected)")
print("   user-99 -> plan=free")
print("\nCompaction runs asynchronously; in a short demo you usually still see all versions until it triggers.")


---

## 5️⃣ Replication & `min.insync.replicas`

### 😬 Bad: `replication.factor=1`

One copy of each partition. If that broker dies, the data is gone. Not a durability problem — a **data-loss** problem.

### ✅ Best: replicate across brokers and enforce a quorum on write

In production you typically run **3 brokers** with:

| Setting | Value | Meaning |
|---|---|---|
| `replication.factor` (topic) | `3` | 3 copies of each partition on 3 different brokers |
| `min.insync.replicas` (topic) | `2` | A write is only acknowledged if **at least 2** in-sync replicas have it |
| `acks` (producer) | `all` | Producer waits for all in-sync replicas |

With these settings:
- **One broker can die** and you still accept writes (2 replicas still in sync ≥ `min.insync.replicas`).
- **Two brokers die** and producers get errors instead of silently losing data (fail loud, not silent).
- Combined with `enable.idempotence=True`, this gives exactly-once semantics on the producer side.

> Our single-broker lab can't actually enforce `min.insync.replicas=2` (not enough brokers), so the cell below just shows the config pattern you'd use. Trying to set `replication.factor=3` on a one-broker cluster would fail with `InvalidReplicationFactor` — that failure is a feature, not a bug.


In [ ]:
production_topic_config = {
    'min.insync.replicas': '2',
    'unclean.leader.election.enable': 'false',
}

production_producer_config = {
    **KAFKA_CONFIG,
    'acks': 'all',
    'enable.idempotence': True,
    'retries': 10,
    'max.in.flight.requests.per.connection': 5,
}

print("📋 Production-ready topic config:")
for k, v in production_topic_config.items():
    print(f"   {k} = {v}")
print("\n📋 Production-ready producer config:")
for k, v in production_producer_config.items():
    if k != 'bootstrap.servers':
        print(f"   {k} = {v}")

print("\n💡 Together these give you: no data loss on single-broker failure,")
print("   no silent data loss on double-broker failure, and no duplicates from retries.")


---

## 6️⃣ Consumer Lag Monitoring

**Consumer lag** = `high_watermark` (latest offset) − `committed_offset` (how far the consumer has read).

### 😬 Bad: no monitoring

Your consumer falls behind silently. Users complain "notifications are 3 hours late" before anyone notices. Lag is the **single most important health metric** for a Kafka consumer.

### ✅ Best: measure lag and alert when it grows

Kafka exposes this via the admin API. In production, tools like **Kafka UI**, **Burrow**, **Prometheus JMX exporter**, or **Confluent Control Center** track lag per consumer group and per partition.

The code below shows how to compute it yourself — useful to understand what those tools do under the hood.


In [ ]:
from confluent_kafka import TopicPartition

group = 'lag-demo-group'

ensure_topic('lag-demo', partitions=2)
p = Producer(KAFKA_CONFIG)
for i in range(50):
    p.produce('lag-demo', key=f'k{i%5}'.encode(), value=f'event-{i}'.encode())
p.flush()

c = Consumer({**KAFKA_CONFIG, 'group.id': group, 'auto.offset.reset': 'earliest',
              'enable.auto.commit': True, 'auto.commit.interval.ms': 100})
c.subscribe(['lag-demo'])
read = 0
while read < 10:
    msg = c.poll(2.0)
    if msg is None: continue
    if msg.error(): continue
    read += 1
time.sleep(0.5)  # let the last commit land
c.close()
print(f"✅ Consumer read {read} messages then stopped")

meta = admin.list_topics('lag-demo', timeout=5)
partitions = [TopicPartition('lag-demo', pid) for pid in meta.topics['lag-demo'].partitions]

c2 = Consumer({**KAFKA_CONFIG, 'group.id': group})
committed = c2.committed(partitions, timeout=5)

total_lag = 0
print(f"\n📊 Consumer-group lag for '{group}':")
for tp in committed:
    low, high = c2.get_watermark_offsets(tp, timeout=5)
    committed_off = tp.offset if tp.offset >= 0 else 0
    lag = high - committed_off
    total_lag += lag
    print(f"   partition {tp.partition}: committed={committed_off}, high-watermark={high}, lag={lag}")
c2.close()

print(f"\n🚨 Total lag across all partitions: {total_lag}")
print("💡 Alert when lag exceeds a threshold (e.g., 1000 messages or 5 minutes of events).")


---

## 📝 Key Takeaways

| Topic | Bad | Best |
|---|---|---|
| **Schemas** | Publish any JSON | Validate at the producer; use Schema Registry in production |
| **Throughput** | `flush()` every message | Tune `linger.ms`, `batch.size`, `compression.type` |
| **Errors** | Crash on poison messages | Route to a Dead Letter Queue topic |
| **Retention** | Keep everything forever | Pick `delete` or `compact` per use case |
| **Durability** | `replication.factor=1` | `replication.factor=3`, `min.insync.replicas=2`, `acks=all` |
| **Observability** | No monitoring | Track consumer lag per group / per partition |

### Rule of thumb

> **In production, assume every link fails:** brokers die, networks partition, messages are malformed, consumers crash. Kafka gives you the tools to handle each — you just have to turn them on.

---

## 🎉 You've finished the Kafka Deep Dive!

Go back to notebooks 01–04 and experiment:
- Chain filter → map → aggregate across multiple topics.
- Add schema validation to notebook 1.
- Wrap notebook 3's transactional consumer with a DLQ.

Happy streaming! 🚀
